In [1]:
import os
import pandas as pd
import torch
import spacy

from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    CLIPProcessor,
    CLIPModel
)

In [2]:
device = "cpu"

print("Device yang digunakan:", device)
print("Jumlah CPU thread:", os.cpu_count())

Device yang digunakan: cpu
Jumlah CPU thread: 8


In [15]:
CSV_PATH = "instagram_images.csv"
IMAGE_FOLDER = "instagram_images"
OUTPUT_CSV = "clip-blip.csv"

print("Current folder :", os.getcwd())
print("CSV            :", CSV_PATH)
print("Folder gambar  :", IMAGE_FOLDER)
print("Output         :", OUTPUT_CSV)

Current folder : c:\Users\elsae\.vscode\Documents\city-branding-multimodal\Scraping Data Zefanya (10 Aug )
CSV            : instagram_images.csv
Folder gambar  : instagram_images
Output         : clip-blip.csv


In [16]:
print("CSV ditemukan:", os.path.exists(CSV_PATH))
print("Folder gambar ditemukan:", os.path.exists(IMAGE_FOLDER))

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"CSV tidak ditemukan: {CSV_PATH}"
    )

if not os.path.exists(IMAGE_FOLDER):
    raise FileNotFoundError(
        f"Folder gambar tidak ditemukan: {IMAGE_FOLDER}"
    )

CSV ditemukan: True
Folder gambar ditemukan: True


In [17]:
print("Loading BLIP...")

blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model = blip_model.to(device)
blip_model.eval()

print("BLIP berhasil dimuat.")

Loading BLIP...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BLIP berhasil dimuat.


In [18]:
print("Loading CLIP...")

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)

clip_model = clip_model.to(device)
clip_model.eval()

print("CLIP berhasil dimuat.")

Loading CLIP...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP berhasil dimuat.


In [19]:
nlp = spacy.load("en_core_web_sm")

print("spaCy berhasil dimuat.")

spaCy berhasil dimuat.


In [20]:
def generate_caption(image):

    inputs = blip_processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():

        output = blip_model.generate(
            **inputs,
            max_new_tokens=50
        )

    caption = blip_processor.decode(
        output[0],
        skip_special_tokens=True
    )

    return caption.strip()

In [21]:
def extract_candidates(caption):

    doc = nlp(caption)

    candidates = []

    for chunk in doc.noun_chunks:

        candidate = chunk.text.lower().strip()

        # Hilangkan article
        for article in ["a ", "an ", "the "]:

            if candidate.startswith(article):
                candidate = candidate[len(article):]

        candidate = candidate.strip()

        # Hindari kosong dan duplikat
        if (
            candidate
            and candidate not in candidates
            and len(candidate) > 2
        ):
            candidates.append(candidate)

    return candidates

In [22]:
def rank_candidates(image, candidates):

    if len(candidates) == 0:
        return []

    prompts = [
        f"a photo of {candidate}"
        for candidate in candidates
    ]

    inputs = clip_processor(
        text=prompts,
        images=image,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():

        outputs = clip_model(**inputs)

        logits = outputs.logits_per_image

        probabilities = logits.softmax(
            dim=1
        )[0]

    results = []

    for candidate, probability in zip(
        candidates,
        probabilities
    ):

        results.append({
            "scene": candidate,
            "score": probability.item()
        })

    results = sorted(
        results,
        key=lambda x: x["score"],
        reverse=True
    )

    return results

In [23]:
def process_image(image_path):

    image = Image.open(
        image_path
    ).convert("RGB")

    # ==============================
    # BLIP
    # ==============================
    caption_blip = generate_caption(
        image
    )

    # ==============================
    # spaCy
    # ==============================
    candidates = extract_candidates(
        caption_blip
    )

    # Kalau tidak mendapatkan kandidat
    if len(candidates) == 0:

        return (
            caption_blip,
            "",
            "",
            ""
        )

    # ==============================
    # CLIP
    # ==============================
    results = rank_candidates(
        image,
        candidates
    )

    top_scenes = [
        result["scene"]
        for result in results[:3]
    ]

    # Kalau kandidat kurang dari tiga
    while len(top_scenes) < 3:
        top_scenes.append("")

    return (
        caption_blip,
        top_scenes[0],
        top_scenes[1],
        top_scenes[2]
    )

In [24]:
df = pd.read_csv(
    CSV_PATH,
    dtype={
        "FileName": str
    }
)

print("Jumlah data:", len(df))
print("Jumlah kolom:", len(df.columns))

display(df.head())

Jumlah data: 2689
Jumlah kolom: 24


,caption,owner/fullName,owner/id,owner/isPrivate,owner/isVerified,owner/profilePicUrl,owner/username,url,likeCount,location,...,location/lng,location/name,location/slug,createdAt,id,image/height,image/url,image/width,FileName,DownloadStatus
0,[ Yuk Travelling Asyik Bersama Kami Restu Bumi...,RESTU BUMI ADVENTURE | Travel Consultant,2997670801,False,True,https://scontent-cph2-1.cdninstagram.com/v/t51...,restubumiadventure,https://www.instagram.com/p/CwwPVBmPzHQ/,27.0,NaN,...,NaN,NaN,NaN,2023-09-04T02:49:53.000Z,3.184110e+18,1350,https://scontent-cph2-1.cdninstagram.com/v/t51...,1080,image_000003.jpg,success
1,Let the fun roll!\nAjak temen-temen kamu buat ...,Greencanyon by Kedas Resort,73357119285,False,False,https://scontent-cph2-1.cdninstagram.com/v/t51...,greencanyonbykedasresort,https://www.instagram.com/p/DXv1jH8jpWO/,15.0,NaN,...,105.713648,Minang Rua Green Canyon,Minang Rua Green Canyon,2026-04-30T07:28:59.000Z,3.886560e+18,1350,https://scontent-cph2-1.cdninstagram.com/v/t51...,1080,image_000011.jpg,success
2,@navaracitypark \n#lampunghits \n#lampunghitsk...,misnawati iis,66845543644,False,False,https://scontent-cph2-1.cdninstagram.com/v/t51...,misnawatiiis24,https://www.instagram.com/p/DXwokcmE72R/,2.0,NaN,...,NaN,NaN,NaN,2026-04-30T14:54:51.000Z,3.886780e+18,1440,https://scontent-cph2-1.cdninstagram.com/v/t51...,1080,image_000012.jpg,success
3,#eksploreindonesia#pulausebukukecil#lampung#la...,Epik Hartuti,6997736428,False,False,https://scontent-cph2-1.cdninstagram.com/v/t51...,epik_hartuti,https://www.instagram.com/p/DXn2bNTkXSV/,0.0,NaN,...,NaN,NaN,NaN,2026-04-27T05:02:43.000Z,3.884310e+18,1440,https://scontent-cph2-1.cdninstagram.com/v/t51...,1080,image_000013.jpg,success
4,Pantai Minang Rua 🏝️🛵 #wisatalampungselatan #t...,NaN,14623907147,False,False,https://scontent-cph2-1.cdninstagram.com/v/t51...,bizzyboyz__,https://www.instagram.com/p/DbPGngQD1ga/,11.0,NaN,...,NaN,NaN,NaN,2026-07-26T00:28:46.000Z,3.949400e+18,1440,https://scontent-cph2-1.cdninstagram.com/v/t51...,1080,image_000016.jpg,success


In [25]:
if "FileName" not in df.columns:

    raise ValueError(
        "Kolom FileName tidak ditemukan dalam CSV."
    )

print("Kolom FileName ditemukan.")

Kolom FileName ditemukan.


In [26]:
image_files = os.listdir(
    IMAGE_FOLDER
)

print(
    "Jumlah file dalam folder instagram_images:",
    len(image_files)
)

print(
    "Jumlah baris CSV:",
    len(df)
)

Jumlah file dalam folder instagram_images: 2689
Jumlah baris CSV: 2689


In [27]:
folder_files = set(
    os.listdir(IMAGE_FOLDER)
)

csv_files = set(
    df["FileName"]
    .dropna()
    .astype(str)
    .str.strip()
)

matched = csv_files.intersection(
    folder_files
)

missing = csv_files - folder_files

print("FileName CSV          :", len(csv_files))
print("Cocok dengan folder   :", len(matched))
print("Tidak ditemukan       :", len(missing))

FileName CSV          : 2689
Cocok dengan folder   : 2689
Tidak ditemukan       : 0


In [28]:
for column in [
    "caption_blip",
    "top_1",
    "top_2",
    "top_3"
]:

    if column not in df.columns:
        df[column] = ""

In [29]:
TEST_COUNT = 3

for i in range(
    min(TEST_COUNT, len(df))
):

    filename = str(
        df.iloc[i]["FileName"]
    ).strip()

    image_path = os.path.join(
        IMAGE_FOLDER,
        filename
    )

    print("\n============================")
    print("File:", filename)
    print("============================")

    if not os.path.exists(image_path):

        print("File tidak ditemukan.")
        continue

    try:

        caption_blip, top1, top2, top3 = process_image(
            image_path
        )

        print("Caption BLIP :", caption_blip)
        print("Top 1        :", top1)
        print("Top 2        :", top2)
        print("Top 3        :", top3)

    except Exception as e:

        print("ERROR:", e)


File: image_000003.jpg
Caption BLIP : a group of people posing with an elephant
Top 1        : elephant
Top 2        : group
Top 3        : people

File: image_000011.jpg
Caption BLIP : a man in a yellow raft floating on the water
Top 1        : yellow raft
Top 2        : water
Top 3        : man

File: image_000012.jpg
Caption BLIP : a woman standing in front of a ferris
Top 1        : ferris
Top 2        : front
Top 3        : woman


In [30]:
SAVE_EVERY = 25

for position, (index, row) in enumerate(
    tqdm(
        df.iterrows(),
        total=len(df),
        desc="BLIP + CLIP"
    ),
    start=1
):

    filename = str(
        row["FileName"]
    ).strip()

    image_path = os.path.join(
        IMAGE_FOLDER,
        filename
    )

    # ==============================================
    # Jika file tidak ditemukan
    # ==============================================
    if not os.path.exists(image_path):

        df.at[
            index,
            "caption_blip"
        ] = "FILE_NOT_FOUND"

        continue

    try:

        caption_blip, top1, top2, top3 = process_image(
            image_path
        )

        df.at[
            index,
            "caption_blip"
        ] = caption_blip

        df.at[
            index,
            "top_1"
        ] = top1

        df.at[
            index,
            "top_2"
        ] = top2

        df.at[
            index,
            "top_3"
        ] = top3

    except UnidentifiedImageError:

        print(
            f"\nGambar rusak/tidak dikenali: {filename}"
        )

        df.at[
            index,
            "caption_blip"
        ] = "INVALID_IMAGE"

    except Exception as e:

        print(
            f"\nERROR pada {filename}: {e}"
        )

        df.at[
            index,
            "caption_blip"
        ] = "ERROR"

    # ==============================================
    # Autosave
    # ==============================================
    if position % SAVE_EVERY == 0:

        df.to_csv(
            OUTPUT_CSV,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"\nAutosave: {position}/{len(df)}"
        )

BLIP + CLIP:   0%|          | 0/2689 [00:00<?, ?it/s]


Autosave: 25/2689

Autosave: 50/2689

Autosave: 75/2689

Autosave: 100/2689

Autosave: 125/2689

Autosave: 150/2689

Autosave: 175/2689

Autosave: 200/2689

Autosave: 225/2689

Autosave: 250/2689

Autosave: 275/2689

Autosave: 300/2689

Autosave: 325/2689

Autosave: 350/2689

Autosave: 375/2689

Autosave: 400/2689

Autosave: 425/2689

Autosave: 450/2689

Autosave: 475/2689

Autosave: 500/2689

Autosave: 525/2689

Autosave: 550/2689

Autosave: 575/2689

Autosave: 600/2689

Autosave: 625/2689

Autosave: 650/2689

Autosave: 675/2689

Autosave: 700/2689

Autosave: 725/2689

Autosave: 750/2689

Autosave: 775/2689

Autosave: 800/2689

Autosave: 825/2689

Autosave: 850/2689

Autosave: 875/2689

Autosave: 900/2689

Autosave: 925/2689

Autosave: 950/2689

Autosave: 975/2689

Autosave: 1000/2689

Autosave: 1025/2689

Autosave: 1050/2689

Autosave: 1075/2689

Autosave: 1100/2689

Autosave: 1125/2689

Autosave: 1150/2689

Autosave: 1175/2689

Autosave: 1200/2689

Autosave: 1225/2689

Autosave: 12

In [31]:
df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("==============================")
print("SELESAI")
print("==============================")

print(
    "Output tersimpan:",
    OUTPUT_CSV
)

print(
    "Jumlah data:",
    len(df)
)

SELESAI
Output tersimpan: clip-blip.csv
Jumlah data: 2689


In [32]:
display(
    df[
        [
            "FileName",
            "caption",
            "caption_blip",
            "top_1",
            "top_2",
            "top_3"
        ]
    ].head(20)
)

,FileName,caption,caption_blip,top_1,top_2,top_3
0,image_000003.jpg,[ Yuk Travelling Asyik Bersama Kami Restu Bumi...,a group of people posing with an elephant,elephant,group,people
1,image_000011.jpg,Let the fun roll!\nAjak temen-temen kamu buat ...,a man in a yellow raft floating on the water,yellow raft,water,man
2,image_000012.jpg,@navaracitypark \n#lampunghits \n#lampunghitsk...,a woman standing in front of a ferris,ferris,front,woman
3,image_000013.jpg,#eksploreindonesia#pulausebukukecil#lampung#la...,two women in the water at the beach,beach,two women,water
4,image_000016.jpg,Pantai Minang Rua 🏝️🛵 #wisatalampungselatan #t...,a group of people posing for a picture,group,picture,people
5,image_000017.jpg,cukup dengan beberapa foto kita bisa buktikan ...,a man standing on a bridge in front of a water...,waterfall,bridge,man
6,image_000019.jpg,"i don't know how to thank you, but i am so gla...",a col with a col of a couple,couple,col,
7,image_000022.jpg,Seni menikmati ciptaanMU 🤍🌊\n.\n.\n#wisatalamp...,a woman in a white coat sitting on rocks near ...,woman,ocean,white coat
8,image_000029.jpg,Salah satu pantai yg dekat dengan Pelabuhan Ba...,the sun is setting over the river,river,sun,
9,image_000032.jpg,Continue…\nKarena sudah malam setelah nengok s...,a woman and child walking on the beach,beach,child,woman
